In [1]:
import cv2
import numpy as np
import pytesseract
import re

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def limpar_texto_extra(texto):
    # Corrige confusões comuns do Tesseract em DANFEs
    return texto.replace('O', '0').replace('o', '0').replace('I', '1').replace('|', '')

def processar_danfe(caminho_imagem):
    img = cv2.imread(caminho_imagem)
    if img is None: return "Erro ao carregar", {}

    # 1. AUMENTAR A IMAGEM (Essencial para textos pequenos em tabelas)
    # Vamos aumentar em 3x para dar nitidez aos números
    img = cv2.resize(img, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

    # 2. PRÉ-PROCESSAMENTO
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Brilho e Contraste (ajuda a separar o texto do fundo cinza)
    gray = cv2.convertScaleAbs(gray, alpha=1.2, beta=0)
    
    # Binarização (Preto no Branco)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 3. OCR (PSM 3 para detectar blocos espalhados)
    texto = pytesseract.image_to_string(thresh, lang='por', config='--psm 3')
    texto_limpo = limpar_texto_extra(texto)

    # 4. REGEX ESPECÍFICOS PARA ESTA NOTA
    dados = {
        # CNPJ Emitente (está abaixo do campo DANFE)
        "cnpj_emitente": re.findall(r'(\d{2}\.?\d{3}\.?\d{3}/\d{4}-?\d{2})', texto_limpo),
        
        # Valor Total (O campo fica na direita, "VALOR TOTAL DA NOTA")
        # Vamos buscar o valor que aparece perto de "370,80" que está na sua nota
        "valor_total": re.findall(r'(?:TOTAL|NOTA)\s*(?:R\$)?\s*[:\.]?\s*([\d\.,]+)', texto_limpo, re.IGNORECASE),
        
        # Chave de Acesso (44 dígitos em cima do código de barras)
        "chave_acesso": ["".join(re.findall(r'\d', c)) for c in re.findall(r'[\d\s]{40,65}', texto_limpo) if len("".join(re.findall(r'\d', c))) == 44]
    }

    return texto, dados

# --- Execução ---
caminho = r"C:\Users\LeonardoCampos\Downloads\WhatsApp Image 2026-04-02 at 11.09.35.jpeg"
texto_bruto, info = processar_danfe(caminho)

print("=== RESULTADO DA ANÁLISE ===")
# Exibindo os dados de forma limpa
cnpj = info['cnpj_emitente'][0] if info['cnpj_emitente'] else "Não encontrado"
valor = info['valor_total'][-1] if info['valor_total'] else "Não encontrado"
chave = info['chave_acesso'][0] if info['chave_acesso'] else "Não encontrada"

print(f"CNPJ Emitente: {cnpj}")
print(f"Chave de Acesso: {chave}")
print(f"Valor Total: R$ {valor}")

# Se quiser ver o que o OCR leu para ajustar o Regex:
print(texto_bruto)

ModuleNotFoundError: No module named 'cv2'